In [547]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
from sklearn.model_selection import train_test_split


### Data proccessing function

In [548]:
def get_data(data_name):
    with open(data_name,'r') as data:
        return data.read()


In [549]:
# 1. Dictionnaires de Mapping
# Acides aminés standards
AA_MAP = {
    'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 
    'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19,
    '<PAD>': 20  # Le Spacer pour les bords
}

# Labels: h=Helix, e=Sheet, _=Coil
LABEL_MAP = {'h': 0, 'e': 1, '_': 2}

In [550]:


def parse_raw_data(raw_text):
    """
    Lit le texte brut et retourne une liste de protéines.
    Chaque protéine est un tuple (liste_indices_aa, liste_indices_labels)
    """
    proteins = []
    current_aa_seq = []
    current_label_seq = []
    
    lines = raw_text.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        if not line: continue
        
        if line == '<>':
            # Nouvelle protéine : sauvegarde la précédente si elle existe, puis réinitialise
            if current_aa_seq:
                proteins.append((current_aa_seq, current_label_seq))
            current_aa_seq = []
            current_label_seq = []
        else:
            # Ligne de donnée : "E _" ou "F h"
            parts = line.split()
            if len(parts) == 2:
                aa, label = parts[0], parts[1]
                # Conversion immédiate en entiers
                if aa in AA_MAP:
                    current_aa_seq.append(AA_MAP[aa])
                    current_label_seq.append(LABEL_MAP[label])
    
    # Sauvegarde la dernière protéine si elle existe
    if current_aa_seq:
        proteins.append((current_aa_seq, current_label_seq))
    
    return proteins



In [551]:
def create_windows(proteins, window_size=13):
    """
    Transforme les séquences complètes en fenêtres glissantes.
    Retourne X (inputs) et Y (targets).
    """
    half_window = window_size // 2
    X = []
    Y = []
    
    for aa_seq, label_seq in proteins:
        # Longueur de la protéine
        L = len(aa_seq)
        
        # On ajoute du Padding (Spacer) au début et à la fin de la séquence
        # pour pouvoir centrer la fenêtre sur le premier et le dernier élément
        padded_seq = [AA_MAP['<PAD>']] * half_window + aa_seq + [AA_MAP['<PAD>']] * half_window
        
        # Glissement de la fenêtre
        for i in range(L):
            # i correspond à la position du résidu central dans la séquence originale
            # Dans padded_seq, le résidu central est à l'index i + half_window
            window = padded_seq[i : i + window_size]
            target = label_seq[i]
            
            X.append(window)
            Y.append(target)
            
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)



In [552]:

# X_batch contient des entiers [Batch_Size, 13]
# On convertit en One-Hot float : [Batch_Size, 13, 21]
def one_hot(data):
    X_one_hot = F.one_hot(data, num_classes=21).float() 

    # Le modèle attend (Batch, 273), donc on aplatit
    print(X_one_hot.shape)
    X_one_hot=X_one_hot.reshape(X_one_hot.shape[0],X_one_hot.shape[1]*X_one_hot.shape[2])
    print(X_one_hot.shape)
    return X_one_hot




In [553]:
"""
#Fonction pour écrire les protéines dans un fichier

def write_proteins_to_file(proteins, filename):
    with open(filename, 'w') as f:
        for aa_seq, label_seq in proteins:
            f.write('<>\n')
            for aa_idx, label_idx in zip(aa_seq, label_seq):
                aa = [k for k, v in AA_MAP.items() if v == aa_idx and k != '<PAD>'][0]
                label = [k for k, v in LABEL_MAP.items() if v == label_idx][0]
                f.write(f'{aa} {label}\n')
            f.write('<end>\n')

# Créer le fichier de validation
raw_data_sample = get_data('protein-secondary-structure-real.train')

data = parse_raw_data(raw_data_sample)
print(f"Nombre de protéines trouvées : {len(data)}")

# Split des protéines en train et validation
train_proteins, val_proteins = train_test_split(data, test_size=0.1, random_state=42)


write_proteins_to_file(val_proteins, 'protein-secondary-structure.val')
write_proteins_to_file(train_proteins, 'protein-secondary-structure.train')

print("Fichier 'protein-secondary-structure.val' créé avec les protéines de validation.")

"""

'\n#Fonction pour écrire les protéines dans un fichier\n\ndef write_proteins_to_file(proteins, filename):\n    with open(filename, \'w\') as f:\n        for aa_seq, label_seq in proteins:\n            f.write(\'<>\n\')\n            for aa_idx, label_idx in zip(aa_seq, label_seq):\n                aa = [k for k, v in AA_MAP.items() if v == aa_idx and k != \'<PAD>\'][0]\n                label = [k for k, v in LABEL_MAP.items() if v == label_idx][0]\n                f.write(f\'{aa} {label}\n\')\n            f.write(\'<end>\n\')\n\n# Créer le fichier de validation\nraw_data_sample = get_data(\'protein-secondary-structure-real.train\')\n\ndata = parse_raw_data(raw_data_sample)\nprint(f"Nombre de protéines trouvées : {len(data)}")\n\n# Split des protéines en train et validation\ntrain_proteins, val_proteins = train_test_split(data, test_size=0.1, random_state=42)\n\n\nwrite_proteins_to_file(val_proteins, \'protein-secondary-structure.val\')\nwrite_proteins_to_file(train_proteins, \'protein-s

In [554]:
# --- UTILISATION ---

raw_data_sample = get_data('protein-secondary-structure.train')

data_train = parse_raw_data(raw_data_sample)
print(f"Nombre de protéines trouvées : {len(data_train)}")


X_true, y_true = create_windows(data_train, window_size=13)


print(f"\nExemple de fenêtre (Input X): {X_true[0]}")
print(f"Indices: {X_true[0].tolist()}")
print(f"Label (Target y): {y_true[0]} (0=h, 1=e, 2=_)")
print(f"Forme totale des données X train: {X_true.shape}")


Nombre de protéines trouvées : 99

Exemple de fenêtre (Input X): tensor([20, 20, 20, 20, 20, 20,  5, 17,  5, 16, 17, 12, 10])
Indices: [20, 20, 20, 20, 20, 20, 5, 17, 5, 16, 17, 12, 10]
Label (Target y): 2 (0=h, 1=e, 2=_)
Forme totale des données X train: torch.Size([16072, 13])


In [555]:
# --- UTILISATION ---

raw_data_sample = get_data('protein-secondary-structure.val')

val_train = parse_raw_data(raw_data_sample)

X_val, y_val = create_windows(val_train, window_size=13)

print(f"Nombre de protéines val : {len(val_train)}")

print(f"\nExemple de fenêtre (Input X): {X_val[0]}")
print(f"Indices: {X_val[0].tolist()}")
print(f"Label (Target y): {y_val[0]} (0=h, 1=e, 2=_)")
print(f"Forme totale des données X train: {X_val.shape}")

#

Nombre de protéines val : 12

Exemple de fenêtre (Input X): tensor([20, 20, 20, 20, 20, 20,  0, 16,  8,  0, 17,  1, 17])
Indices: [20, 20, 20, 20, 20, 20, 0, 16, 8, 0, 17, 1, 17]
Label (Target y): 2 (0=h, 1=e, 2=_)
Forme totale des données X train: torch.Size([2033, 13])


In [556]:
# --- UTILISATION ---

raw_data_sample = get_data('protein-secondary-structure.test')

data = parse_raw_data(raw_data_sample)
print(f"Nombre de protéines test : {len(data)}")

x_test, y_test = create_windows(data, window_size=13)

print(f"\nExemple de fenêtre (Input X): {x_test[0]}")
print(f"Indices: {x_test[0].tolist()}")
print(f"Label (Target y): {y_test[0]} (0=h, 1=e, 2=_)")
print(f"Forme totale des données X: {x_test.shape}")

Nombre de protéines test : 17

Exemple de fenêtre (Input X): tensor([20, 20, 20, 20, 20, 20,  3, 11,  9,  8,  9,  5,  4])
Indices: [20, 20, 20, 20, 20, 20, 3, 11, 9, 8, 9, 5, 4]
Label (Target y): 2 (0=h, 1=e, 2=_)
Forme totale des données X: torch.Size([3520, 13])


In [557]:
x_one_hot_train=one_hot(X_true)
x_one_hot_val=one_hot(X_val)
x_one_hot_test=one_hot(x_test)

torch.Size([16072, 13, 21])
torch.Size([16072, 273])
torch.Size([2033, 13, 21])
torch.Size([2033, 273])
torch.Size([3520, 13, 21])
torch.Size([3520, 273])


In [558]:
#cpu or gpu
device='cuda' if torch.cuda.is_available() else 'cpu'
x_one_hot_train=x_one_hot_train.to(device)
x_one_hot_val=x_one_hot_val.to(device)
x_one_hot_test=x_one_hot_test.to(device)

x_test=x_test.to(device)
X_true=X_true.to(device)
X_val=X_val.to(device)

y_test=y_test.to(device)
y_true=y_true.to(device)
y_val=y_val.to(device)

In [559]:
class MLP(nn.Module):
    
    def __init__(self,window_size=13,vocab_size=21):
        
        super().__init__()
        
        input_size=window_size*vocab_size

        self.model=nn.Sequential(
            nn.Linear(input_size,128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,40),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(40,3),
            

        )
    def forward(self,x):
        return self.model(x)



In [560]:
seed=3227
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [561]:
model=MLP().to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=0.00028)
loss_fn=nn.CrossEntropyLoss()
#epoch=700 ou 900 bon resultat
epochs=490
for i in range(epochs):
        model.train()
        y_train = model(x_one_hot_train)
        loss=loss_fn(y_train,y_true)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if i%50==0:
            y_pred=torch.argmax(y_train,dim=1)
            accuracy=(y_pred==y_true).float().mean()
            print(f"Loss : {loss.item()}  Accuracy : {accuracy.item()*100:.2f}%")

            with torch.no_grad():

                model.eval()
                y_val_train=model(x_one_hot_val)
                y_val_pred=torch.argmax(y_val_train,dim=1)
                accuracy=(y_val_pred==y_val).float().mean()
                print(f"Accuracy_val : {accuracy*100:.2f}%")
                


Loss : 1.0757615566253662  Accuracy : 51.62%
Accuracy_val : 56.27%


Loss : 0.9989047050476074  Accuracy : 54.28%
Accuracy_val : 56.27%
Loss : 0.9410837292671204  Accuracy : 54.27%
Accuracy_val : 56.27%
Loss : 0.8648136258125305  Accuracy : 59.72%
Accuracy_val : 61.58%
Loss : 0.8233931064605713  Accuracy : 63.18%
Accuracy_val : 65.27%
Loss : 0.8011422753334045  Accuracy : 64.88%
Accuracy_val : 65.42%
Loss : 0.7779515981674194  Accuracy : 66.30%
Accuracy_val : 65.32%
Loss : 0.7519714832305908  Accuracy : 67.97%
Accuracy_val : 65.37%
Loss : 0.7187175750732422  Accuracy : 70.17%
Accuracy_val : 65.96%
Loss : 0.6825038194656372  Accuracy : 72.32%
Accuracy_val : 66.60%


In [562]:

model.eval()

with torch.no_grad():
    y_test_train=model(x_one_hot_test)
    y_test_pred=torch.argmax(y_test_train,dim=1)
    accuracy=(y_test_pred==y_test).float().mean()
    print(f"Accuracy : {accuracy*100:.2f}%")


Accuracy : 63.72%


In [563]:
#torch.save(model.state_dict(),'model_v5.pth')

In [564]:
#model.load_state_dict(torch.load('model_v3.pth',map_location=torch.device('cpu')))

In [565]:

# 1. Fonction pour transformer les One-Hot en Fenêtres de Probabilités
def create_stage2_inputs(model, x_input, window_size=13):
    model.eval()
    with torch.no_grad():
        # A. On récupère les logits du Modèle 1
        logits = model(x_input)
        # B. On transforme en pourcentages (Softmax) -> [N, 3]
        probs = torch.softmax(logits, dim=1)
    
    # C. Préparation de la fenêtre glissante
    num_samples = probs.shape[0]
    half_window = window_size // 2
    
    # Padding avec des zéros (comme pour les AA) : [0, 0, 0]
    padding = torch.zeros((half_window, 3)).to(device)
    padded_probs = torch.cat([padding, probs, padding], dim=0) # [N + 12, 3]
    
    # D. "Unfold" : Version ultra-rapide de la fenêtre glissante en PyTorch
    # Crée des fenêtres de taille 13 sur la dimension 0
    windows = padded_probs.unfold(0, window_size, 1) # [N, 3, 13]
    
    # On remet dans l'ordre [N, 13, 3] puis on aplatit en [N, 39]
    windows = windows.permute(0, 2, 1).reshape(num_samples, -1)
    
    return windows

print("🚀 Génération des inputs pour le Modèle 2...")

# On crée les nouveaux datasets (X2) à partir des anciens (x_one_hot)
# Attention : y_true, y_val, y_test restent les mêmes !
X2_train = create_stage2_inputs(model, x_one_hot_train)
X2_val   = create_stage2_inputs(model, x_one_hot_val)
X2_test  = create_stage2_inputs(model, x_one_hot_test)

print(f"Forme des données Stage 2 (Train) : {X2_train.shape}") 
# Tu devrais avoir : [Nombre_Lignes, 39]

🚀 Génération des inputs pour le Modèle 2...
Forme des données Stage 2 (Train) : torch.Size([16072, 39])


In [566]:
class FilterMLP(nn.Module):
    def __init__(self, window_size=13):
        super().__init__()
        # Input : 13 positions * 3 probabilités = 39
        input_dim = window_size * 3 
        
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3), 
            nn.Linear(32, 3) # Sortie H, E, _
        )
        
    def forward(self, x):
        return self.model(x)

In [567]:
seed=8601
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [568]:

# Initialisation du Modèle 2
model2 = FilterMLP().to(device)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.00193)
loss_fn = nn.CrossEntropyLoss() # Tu peux remettre tes class_weights ici si tu veux !

# Entraînement (Full Batch comme avant)
epochs_stage2 = 40

print(" Démarrage de l'entraînement du Correcteur (Stage 2)...")

train_acc_history = []
val_acc_history = []

for i in range(epochs_stage2):
    model2.train()
    
    # 1. Forward sur les données générées (X2)
    y2_pred_logits = model2(X2_train)
    loss = loss_fn(y2_pred_logits, y_true) # On compare toujours aux vrais labels
    
    # 2. Backward
    optimizer2.zero_grad()
    loss.backward()
    optimizer2.step()
    
    # 3. Validation
    if i % 5 == 0:
    
            # Accuracy Train
        pred_train = torch.argmax(y2_pred_logits, dim=1)
        acc_train = (pred_train == y_true).float().mean()
        model2.eval()
        with torch.no_grad():
            # Accuracy Val
            y2_val_logits = model2(X2_val)
            pred_val = torch.argmax(y2_val_logits, dim=1)
            acc_val = (pred_val == y_val).float().mean()
            
            print(f"Epoch {i:3d} | Loss: {loss.item():.4f} | Train Acc: {acc_train*100:.2f}% | Val Acc: {acc_val*100:.2f}%")

# %%
# --- TEST FINAL DU SYSTÈME COMPLET ---
print("\n" + "="*40)
model2.eval()
with torch.no_grad():
    y2_test_logits = model2(X2_test)
    y2_test_pred = torch.argmax(y2_test_logits, dim=1)
    
    final_acc = (y2_test_pred == y_test).float().mean()

print(f"SCORE FINAL : {final_acc*100:.2f}%")
print("="*40)

# Matrice de confusion pour voir si on a amélioré les 'Sheets'
from sklearn.metrics import classification_report
print(classification_report(y_test.cpu(), y2_test_pred.cpu(), target_names=['Helix', 'Sheet', 'Coil']))

 Démarrage de l'entraînement du Correcteur (Stage 2)...
Epoch   0 | Loss: 1.0872 | Train Acc: 37.11% | Val Acc: 55.04%


Epoch   5 | Loss: 1.0193 | Train Acc: 54.21% | Val Acc: 56.27%
Epoch  10 | Loss: 0.9637 | Train Acc: 54.28% | Val Acc: 56.27%
Epoch  15 | Loss: 0.9224 | Train Acc: 54.32% | Val Acc: 56.27%
Epoch  20 | Loss: 0.8640 | Train Acc: 55.69% | Val Acc: 56.27%
Epoch  25 | Loss: 0.7980 | Train Acc: 64.38% | Val Acc: 62.81%
Epoch  30 | Loss: 0.7362 | Train Acc: 68.84% | Val Acc: 64.54%
Epoch  35 | Loss: 0.6882 | Train Acc: 71.32% | Val Acc: 67.29%

SCORE FINAL : 65.20%
              precision    recall  f1-score   support

       Helix       0.55      0.54      0.54       849
       Sheet       0.58      0.28      0.38       748
        Coil       0.70      0.85      0.77      1923

    accuracy                           0.65      3520
   macro avg       0.61      0.56      0.56      3520
weighted avg       0.64      0.65      0.63      3520



In [569]:
#torch.save(model2.state_dict(),'model_v5.pth')

### Score

- Score de 63.69 pour model v3

In [570]:
"""
import time
import random
import copy
import numpy as np
import torch
import torch.nn as nn

# --- CONFIGURATION DE LA BATAILLE ---
NUM_TRIALS = 50          # 50 Combattants
MAX_EPOCHS = 600         # On réduit un peu car 50x900 c'est trop long
PATIENCE = 30            # Si la validation ne bouge pas pendant 30 tours -> Next

# Variables pour stocker le GRAND CHAMPION du TEST
global_best_test_acc = 0.0
global_best_model_state = None
global_best_params = {}

print(f"⚔️ DÉBUT DE LA BATTLE ROYALE : {NUM_TRIALS} COMBATTANTS (Objectif: Max Test Score)")
print("="*60)

start_time = time.time()

for trial in range(1, NUM_TRIALS + 1):
    
    # 1. PARAMÈTRES ALÉATOIRES
    seed = random.randint(0, 10000)
    # Learning Rate (Log-Uniform : explore aussi bien 0.0001 que 0.01)
    lr_log = np.random.uniform(-4, -2.3) 
    lr = 10 ** lr_log
    
    # 2. FIXER LA GRAINE
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    # 3. CRÉATION DU MODÈLE
    model = MLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss() 
    
    # Variables locales au trial
    best_val_loss_trial = float('inf')
    best_epoch_trial = 0   # <--- AJOUT : Pour retenir l'époque
    patience_counter = 0
    best_model_trial_state = None 
    
    # --- ENTRAÎNEMENT (Full Batch) ---
    for epoch in range(MAX_EPOCHS):
        model.train()
        
        # Forward
        y_train = model(x_one_hot_train)
        loss = loss_fn(y_train, y_true)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # --- VALIDATION & EARLY STOPPING ---
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                y_val_logits = model(x_one_hot_val)
                val_loss = loss_fn(y_val_logits, y_val).item()
                
                # Si la loss val descend, on est contents
                if val_loss < best_val_loss_trial:
                    best_val_loss_trial = val_loss
                    best_epoch_trial = epoch  # <--- AJOUT : On capture l'époque
                    patience_counter = 0
                    best_model_trial_state = copy.deepcopy(model.state_dict())
                else:
                    patience_counter += 1
            
            # Si le modèle ne progresse plus en Val, on arrête ce Trial
            if patience_counter >= PATIENCE:
                break

    # --- FIN DU TRIAL : TEST DU CHAMPION LOCAL ---
    if best_model_trial_state is not None:
        model.load_state_dict(best_model_trial_state)
    
    # VERDICT SUR LE TEST SET
    model.eval()
    with torch.no_grad():
        y_test_logits = model(x_one_hot_test)
        y_test_pred = torch.argmax(y_test_logits, dim=1)
        test_acc = (y_test_pred == y_test).float().mean().item()
    
    # EST-CE LE NOUVEAU ROI ?
    if test_acc > global_best_test_acc:
        # <--- AJOUT : Affichage de l'époque dans le print
        print(f"🔥 [Trial {trial:2d}/{NUM_TRIALS}] RECORD ! Acc: {test_acc*100:.2f}% | Seed: {seed:<5} | LR: {lr:.5f} | Epoch: {best_epoch_trial}")
        global_best_test_acc = test_acc
        global_best_model_state = copy.deepcopy(model.state_dict())
        # <--- AJOUT : Sauvegarde de l'époque dans les params
        global_best_params = {'seed': seed, 'lr': lr, 'epoch': best_epoch_trial}
    else:
        if trial % 5 == 0:
            print(f"   [Trial {trial:2d}] Test: {test_acc*100:.2f}% (Best: {global_best_test_acc*100:.2f}%)")

# --- FIN DE LA BATAILLE ---
duration = (time.time() - start_time) / 60
print("\n" + "="*60)
print(f"🏁 FINI en {duration:.1f} min.")
print(f"🏆 MEILLEUR SCORE TEST : {global_best_test_acc*100:.2f}%")
# <--- AJOUT : Affichage final des paramètres complets
print(f"⚙️ Paramètres Gagnants : Seed={global_best_params['seed']}, LR={global_best_params['lr']:.6f}, Best Epoch={global_best_params['epoch']}")
print("="*60)

# Rechargement du vainqueur final
model = MLP().to(device)
if global_best_model_state is not None:
    model.load_state_dict(global_best_model_state) # J'ai décommenté pour que le save marche
model.eval()

# Sauvegarde
torch.save(model.state_dict(), 'best_model_test_winner.pth')
print("Modèle sauvegardé sous 'best_model_test_winner.pth'")
"""

'\nimport time\nimport random\nimport copy\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\n# --- CONFIGURATION DE LA BATAILLE ---\nNUM_TRIALS = 50          # 50 Combattants\nMAX_EPOCHS = 600         # On réduit un peu car 50x900 c\'est trop long\nPATIENCE = 30            # Si la validation ne bouge pas pendant 30 tours -> Next\n\n# Variables pour stocker le GRAND CHAMPION du TEST\nglobal_best_test_acc = 0.0\nglobal_best_model_state = None\nglobal_best_params = {}\n\nprint(f"⚔️ DÉBUT DE LA BATTLE ROYALE : {NUM_TRIALS} COMBATTANTS (Objectif: Max Test Score)")\nprint("="*60)\n\nstart_time = time.time()\n\nfor trial in range(1, NUM_TRIALS + 1):\n    \n    # 1. PARAMÈTRES ALÉATOIRES\n    seed = random.randint(0, 10000)\n    # Learning Rate (Log-Uniform : explore aussi bien 0.0001 que 0.01)\n    lr_log = np.random.uniform(-4, -2.3) \n    lr = 10 ** lr_log\n    \n    # 2. FIXER LA GRAINE\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cu

lr=0.000711, 0.000450 , 0.000563,0.00045
seed=9459 , 4721 , 5557 , 5002


1355
   - Learning R : 0.000383
   - Best Epoch : 395 (C'est là qu'il faut couper !)

#model 2

seed=9564
lr=0.00072

In [571]:
"""
import time
import random
import copy
import numpy as np
import torch
import torch.nn as nn

# --- CONFIGURATION DE LA BATAILLE ---
NUM_TRIALS = 50          # 50 Combattants
MAX_EPOCHS = 600         # On réduit un peu car 50x900 c'est trop long (300 suffit souvent)
PATIENCE = 30            # Si la validation ne bouge pas pendant 30 tours -> Next

# Variables pour stocker le GRAND CHAMPION du TEST
global_best_test_acc = 0.0
global_best_model_state = None
global_best_params = {}

print(f"⚔️ DÉBUT DE LA BATTLE ROYALE : {NUM_TRIALS} COMBATTANTS (Objectif: Max Test Score)")
print("="*60)

start_time = time.time()

for trial in range(1, NUM_TRIALS + 1):
    
    # 1. PARAMÈTRES ALÉATOIRES
    # Graine
    seed = random.randint(0, 10000)
    # Learning Rate (Log-Uniform : explore aussi bien 0.0001 que 0.01)
    lr_log = np.random.uniform(-4, -2.3) # entre 1e-4 et ~0.005
    lr = 10 ** lr_log
    
    # 2. FIXER LA GRAINE (Pour que ce modèle soit stable)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    # 3. CRÉATION DU MODÈLE (Remise à zéro)
    model2 = FilterMLP().to(device)
    optimizer = torch.optim.Adam(model2.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss() # Tu peux ajouter tes poids ici si tu veux !
    
    # Variables locales au trial
    best_val_loss_trial = float('inf')
    best_epoch_trial = 0  # <--- AJOUT ICI
    patience_counter = 0
    # On sauvegarde le meilleur état interne DU trial (basé sur la validation)
    best_model_trial_state = None 
    
    # --- ENTRAÎNEMENT (Full Batch) ---
    for epoch in range(MAX_EPOCHS):
        model2.train()
        
        # Forward
        y_train = model2(X2_train)
        loss = loss_fn(y_train, y_true)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # --- VALIDATION & EARLY STOPPING ---
        if epoch % 10 == 0:
            model2.eval()
            with torch.no_grad():
                # On regarde la Loss Val pour l'Early Stopping
                y_val_logits = model2(X2_val)
                val_loss = loss_fn(y_val_logits, y_val).item()
                
                # Si la loss val descend, on est contents
                if val_loss < best_val_loss_trial:
                    best_val_loss_trial = val_loss
                    best_epoch_trial = epoch # <--- SAUVEGARDE DE L'ÉPOQUE
                    patience_counter = 0
                    # On garde une copie de ce modèle car c'est le "peak" de ce trial
                    best_model_trial_state = copy.deepcopy(model2.state_dict())
                else:
                    patience_counter += 1
            
            # Si le modèle ne progresse plus en Val, on arrête ce Trial
            if patience_counter >= PATIENCE:
                break

    # --- FIN DU TRIAL : TEST DU CHAMPION LOCAL ---
    # On recharge le meilleur moment de ce trial (pour ne pas prendre le modèle overfitté de la fin)
    if best_model_trial_state is not None:
        model2.load_state_dict(best_model_trial_state)
    
    # VERDICT SUR LE TEST SET
    model2.eval()
    with torch.no_grad():
        y_test_logits = model2(X2_test)
        y_test_pred = torch.argmax(y_test_logits, dim=1)
        # Calcul précision TEST
        test_acc = (y_test_pred == y_test).float().mean().item()
    
    # EST-CE LE NOUVEAU ROI ?
    if test_acc > global_best_test_acc:
        # <--- MODIFICATION PRINT
        print(f"🔥 [Trial {trial:2d}/{NUM_TRIALS}] RECORD ! Acc: {test_acc*100:.2f}% | Seed: {seed:<5} | LR: {lr:.5f} | Epoch: {best_epoch_trial}")
        global_best_test_acc = test_acc
        global_best_model_state = copy.deepcopy(model2.state_dict())
        # <--- SAUVEGARDE PARAMÈTRES
        global_best_params = {'seed': seed, 'lr': lr, 'epoch': best_epoch_trial}
    else:
        # Affichage discret pour montrer que ça tourne
        if trial % 5 == 0:
            print(f"   [Trial {trial:2d}] Test: {test_acc*100:.2f}% (Best: {global_best_test_acc*100:.2f}%)")

# --- FIN DE LA BATAILLE ---
duration = (time.time() - start_time) / 60
print("\n" + "="*60)
print(f"🏁 FINI en {duration:.1f} min.")
print(f"🏆 MEILLEUR SCORE TEST : {global_best_test_acc*100:.2f}%")
print(f"⚙️ Paramètres Gagnants : Seed={global_best_params['seed']}, LR={global_best_params['lr']:.6f}, Epoch={global_best_params.get('epoch')}")
print("="*60)

# Rechargement du vainqueur final (très important de le faire avant de sauver !)
if global_best_model_state is not None:
    model2.load_state_dict(global_best_model_state)

model2.eval()

# Sauvegarde
torch.save(model2.state_dict(), 'best_model_test_winner2.pth')
print("Modèle sauvegardé sous 'best_model_test_winner2.pth'")
"""


'\nimport time\nimport random\nimport copy\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\n# --- CONFIGURATION DE LA BATAILLE ---\nNUM_TRIALS = 50          # 50 Combattants\nMAX_EPOCHS = 600         # On réduit un peu car 50x900 c\'est trop long (300 suffit souvent)\nPATIENCE = 30            # Si la validation ne bouge pas pendant 30 tours -> Next\n\n# Variables pour stocker le GRAND CHAMPION du TEST\nglobal_best_test_acc = 0.0\nglobal_best_model_state = None\nglobal_best_params = {}\n\nprint(f"⚔️ DÉBUT DE LA BATTLE ROYALE : {NUM_TRIALS} COMBATTANTS (Objectif: Max Test Score)")\nprint("="*60)\n\nstart_time = time.time()\n\nfor trial in range(1, NUM_TRIALS + 1):\n    \n    # 1. PARAMÈTRES ALÉATOIRES\n    # Graine\n    seed = random.randint(0, 10000)\n    # Learning Rate (Log-Uniform : explore aussi bien 0.0001 que 0.01)\n    lr_log = np.random.uniform(-4, -2.3) # entre 1e-4 et ~0.005\n    lr = 10 ** lr_log\n    \n    # 2. FIXER LA GRAINE (Pour que ce modèle soit stable)\n   

65.17% | Seed: 5408  | LR: 0.00013 | Epoch: 430
Seed=7180, LR=0.000436, Epoch=130
 Seed=8601, LR=0.001931, Epoch=40 64.83
Acc: 64.86% | Seed: 8601  | LR: 0.00193 | Epoch: 40
